# GEE Index Timelapse — Colab notebook

Build annotated satellite **index animations** (MP4 + GIF) and their inside-vs-outside-AOI
**diagram** from Google Earth Engine, using the
[`gee_animation`](https://github.com/cwinkelmann/GEE_Animation) package.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/feat/evi-index/notebooks/colab_gee_animation.ipynb)

Run the cells top to bottom. This makes **live Earth Engine calls** — you need a Google
account with Earth Engine access and a Cloud project (default `hnee-331218`).

## 1. Setup — install the package (Colab)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    # Public repo -> token-free install. After feat/evi-index merges, switch @feat/evi-index -> @main.
    !pip install -q "git+https://github.com/cwinkelmann/GEE_Animation.git@feat/evi-index#egg=gee_animation[shapefile,notebook]"
    print("Installed gee_animation. If Colab prompts, restart the runtime and re-run from here.")
else:
    print("Not on Colab — using the already-installed package.")

## 2. Parameters — edit me, then run everything below

In [ ]:
PROJECT = "hnee-331218"          # your Earth Engine Cloud project

# Region of interest = the cloud-filtered / charted / outlined area. Default: the
# WNE / Grumsin beech forest, embedded inline so this notebook is self-contained.
REGION_AOI = {"geojson": {"type": "Polygon", "coordinates": [[[13.89772, 52.99623], [13.90363, 52.99701], [13.90531, 52.99496], [13.91167, 52.99416], [13.92405, 52.9947], [13.92273, 52.99019], [13.9229, 52.98848], [13.9199, 52.98432], [13.91519, 52.98327], [13.91233, 52.98141], [13.9124, 52.98343], [13.90985, 52.98233], [13.90325, 52.98277], [13.90312, 52.98238], [13.90568, 52.98101], [13.89522, 52.97846], [13.89179, 52.97705], [13.88442, 52.97676], [13.88167, 52.97758], [13.87563, 52.97629], [13.87216, 52.975], [13.8712, 52.97511], [13.87052, 52.97715], [13.87062, 52.9783], [13.87128, 52.97899], [13.87107, 52.98205], [13.8697, 52.98365], [13.86989, 52.9848], [13.86934, 52.98599], [13.87126, 52.98788], [13.87198, 52.98964], [13.87711, 52.99085], [13.87776, 52.99161], [13.87714, 52.99234], [13.87616, 52.99228], [13.87367, 52.99138], [13.87519, 52.99289], [13.87653, 52.99346], [13.87599, 52.99418], [13.87725, 52.99444], [13.8769, 52.99486], [13.87772, 52.99594], [13.88128, 52.99474], [13.88247, 52.9951], [13.88264, 52.99546], [13.88131, 52.99647], [13.88275, 52.99668], [13.88262, 52.99715], [13.88681, 52.99574], [13.89372, 52.99484], [13.89447, 52.99533], [13.89653, 52.99539], [13.89772, 52.99623]]]}}
# --- alternatives ---
# REGION_AOI = {"bbox": [13.869, 52.975, 13.924, 52.997]}     # a simple rectangle
# On Colab, upload your own GeoJSON or a zipped shapefile:
#   from google.colab import files; up = files.upload()
#   REGION_AOI = {"geojson": next(iter(up))}                  # a .geojson file
#   # for a zipped shapefile: !unzip -o your.zip ; REGION_AOI = {"shapefile": "your.shp"}

BUFFER_M = 1000                  # animation frame = region bbox expanded by this many metres

# Products to build: (sensor, index).
#   sensors: sentinel2, landsat, modis
#   indices: ndvi, evi, ndwi, ndmi, rgb, cir   (any sensor)
#            lst, lst_smw, ecostress           (Landsat only)
#   rgb / cir are composites (no diagram).
PRODUCTS = [("sentinel2", "ndvi"), ("landsat", "lst")]

START, END = "2022-05-01", "2022-09-01"    # [start, end) — widen for a longer timelapse
MAX_CLOUD, REGION_MAX_CLOUD = 60, 40       # scene-level %, region %  (raise for more frames)
FPS, DIMENSIONS = 4, 640

## 3. Authenticate Earth Engine

First run on Colab opens a Google sign-in; paste the token when prompted.

In [ ]:
from gee_animation import auth
auth.init(PROJECT)               # ee.Initialize(project=...); prompts ee.Authenticate() if needed
print("Earth Engine initialised for project:", PROJECT)

## 4. Build the areas of interest

The **region** is your AOI; the **frame** is its bounding box expanded by `BUFFER_M`.

In [ ]:
from gee_animation import aoi
region_geom = aoi.parse(REGION_AOI)
frame_bbox = aoi.frame_bbox_from_region(region_geom, float(BUFFER_M))
frame_aoi = {"bbox": frame_bbox}
frame_geom = aoi.parse(frame_aoi)
print("region area (km^2):", round(region_geom.area().getInfo() / 1e6, 2))
print("frame bbox:", [round(x, 4) for x in frame_bbox])

## 5. Pipeline helper — build one product (animation + diagram)

In [ ]:
import csv
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from gee_animation import charts, collection, compositing, render
from gee_animation.config import RunConfig
from gee_animation.products import INDICES

_SCALE = {"sentinel2": 10, "landsat": 30, "modis": 500}

def _save_diagram(series, csv_path, png_path, index):
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["month", "inside_aoi", "outside_aoi"]); w.writerows(series)
    nan = float("nan")
    months = [r[0] for r in series]
    inside = [nan if r[1] is None else r[1] for r in series]
    outside = [nan if r[2] is None else r[2] for r in series]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(months, inside, marker="o", ms=3, lw=1.2, label="inside AOI")
    ax.plot(months, outside, marker="o", ms=3, lw=1.2, label="outside AOI")
    ax.set_title(f"{index.upper()} - inside vs outside AOI (monthly median)")
    ax.set_ylabel(index.upper())
    step = max(1, len(months) // 16)
    ax.set_xticks(range(0, len(months), step))
    ax.set_xticklabels([months[i] for i in range(0, len(months), step)], rotation=45, ha="right")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(png_path, dpi=110); plt.close(fig)

def build_product(sensor, index):
    name = f"{sensor}_{index}"
    composite = INDICES[index].composite
    out_dir = Path("out") / name
    vmin, vmax, pal = INDICES[index].default_viz
    cfg = RunConfig(
        name=name, project=PROJECT, frame_aoi=frame_aoi, region_aoi=REGION_AOI,
        start=START, end=END, sensor=sensor, index=index, cadence="monthly",
        max_cloud_percent=float(MAX_CLOUD), region_max_cloud_percent=float(REGION_MAX_CLOUD),
        viz_min=vmin, viz_max=vmax, palette=list(pal) if pal else [],
        fps=int(FPS), scale=_SCALE.get(sensor, 30), dimensions=int(DIMENSIONS),
        out_dir=str(out_dir), draw_region=True,
    )
    coll = collection.build(cfg, frame_geom, region_geom)
    frames = compositing.monthly_median(coll, cfg)
    if not frames:
        print(f"[{name}] no frames for {START}..{END} - widen dates or raise cloud thresholds.")
        return None
    paths = render.render(frames, cfg, geometry=frame_geom)
    gif = next((p for p in paths if p.suffix == ".gif"), None)
    mp4 = next((p for p in paths if p.suffix == ".mp4"), None)
    chart = None
    if not composite:
        series = charts.inside_outside_timeseries(frames, region_geom, frame_geom, cfg.scale)
        chart = out_dir / f"{name}_chart.png"
        _save_diagram(series, out_dir / f"{name}_series.csv", chart, index)
    return {"name": name, "frames": frames, "gif": gif, "mp4": mp4,
            "chart": chart, "composite": composite}

## 6. Build & show each product

Each writes `out/<sensor>_<index>/` with the GIF, MP4, per-month PNGs, and (for indices) `_chart.png` + `_series.csv`.

In [ ]:
from IPython.display import Image as IPyImage, Video, display

results = []
for sensor, index in PRODUCTS:
    print(f"\n=== building {sensor} / {index} ===")
    r = build_product(sensor, index)
    if not r:
        continue
    f = r["frames"]
    print(f"{len(f)} frames: {f[0].label} .. {f[-1].label}")
    if r["gif"]:
        display(IPyImage(filename=str(r["gif"])))
    if r["mp4"]:
        display(Video(str(r["mp4"]), embed=True, width=int(DIMENSIONS)))
    if r["chart"]:
        display(IPyImage(filename=str(r["chart"])))
    results.append(r)
print(f"\nBuilt {len(results)} product(s).")

## 7. Download the outputs (Colab)

In [ ]:
if IS_COLAB:
    from google.colab import files
    for r in results:
        for key in ("gif", "mp4", "chart"):
            p = r.get(key)
            if p:
                files.download(str(p))
else:
    print("Outputs are under ./out/<sensor>_<index>/  (gif, mp4, per-month PNGs, chart.png, series.csv)")